# 09 · Base vs post-trained foundation model

**What this notebook decides.** How much of the final capability comes from
general post-training, and how much from task-specific adaptation.

Comparing a fine-tuned model against its own unadapted checkpoint conflates two
sources of capability. Adding the pretraining-only **Base** checkpoint separates
them into three stages:

```text
pretraining  ->  general post-training  ->  text-to-SQL adaptation
```

| | Model | Adapter | Isolates |
|---|---|---|---|
| A | post-trained | none | pretraining + post-training |
| B | post-trained | LoRA | + task adaptation |
| D | base | LoRA | pretraining + task adaptation, no post-training |

**B − D** is the contribution of general post-training, given task adaptation.
**B − A** is the contribution of task adaptation, given post-training.

In [ ]:
import pandas as pd

from qwen_text2sql.reporting import (
    MissingArtifact,
    artifact,
    compare_models,
    environment_report,
    headline_metrics,
    load_records,
    project_root,
)
from qwen_text2sql.reporting.style import use_house_style

ROOT = project_root()
PALETTE = use_house_style("light")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
ROOT

In [ ]:
report = environment_report()
print(f"commit   {report['git_commit']}")
print(f"python   {report['python']}")
print(f"platform {report['platform']}")
pd.Series(report["packages"], name="version").to_frame()

In [ ]:
from qwen_text2sql.config import load_config

post = load_config(ROOT / "configs/qwen35_4b_lora.yaml")
base = load_config(ROOT / "configs/qwen35_4b_base_lora.yaml")

checkpoints = pd.DataFrame(
    {
        "post-trained + LoRA (B)": {
            "model_id": post.model.model_id,
            "rank": post.lora.rank,
            "learning_rate": post.training.learning_rate,
            "epochs": post.training.epochs,
            "seed": post.training.seed,
        },
        "base + LoRA (D)": {
            "model_id": base.model.model_id,
            "rank": base.lora.rank,
            "learning_rate": base.training.learning_rate,
            "epochs": base.training.epochs,
            "seed": base.training.seed,
        },
    }
)
checkpoints["identical"] = checkpoints.iloc[:, 0] == checkpoints.iloc[:, 1]
checkpoints

In [ ]:
confounds = checkpoints[~checkpoints["identical"]].index.drop("model_id", errors="ignore")
if len(confounds):
    print(f"CONFOUNDED: B and D also differ in {list(confounds)}.")
else:
    print("Clean contrast: B and D differ only in the foundation checkpoint.")

### A caveat worth stating before any number appears

The Base checkpoint has not been instruction-tuned, so it may not follow the
prompt format at all before adaptation. A weak **Base unadapted** result is
therefore mostly a statement about instruction-following, not about the
pretrained representation. The comparison that carries information is **B vs D**,
where both have received the same task adaptation.

## The three-way comparison

In [ ]:
RUNS = {
    "post-trained, unadapted (A)": "baseline_records",
    "post-trained + LoRA (B)": "lora_records",
}

available = {name: key for name, key in RUNS.items() if artifact(key).exists}
missing = sorted(set(RUNS) - set(available))
if missing:
    print(f"Not yet available: {missing}")

if len(available) < 2:
    raise MissingArtifact(
        "A decomposition needs at least two evaluated runs, and "
        f"{len(available)} is available. Produce the baseline records via "
        "02_foundation_model_baseline and the adapted records via "
        "05_execution_evaluation, then re-run this notebook."
    )

runs = {name: load_records(artifact(key).require()) for name, key in available.items()}
pd.concat(
    {name: headline_metrics(frame).set_index("metric") for name, frame in runs.items()},
    names=["run"],
).style.format({"estimate": "{:.1%}", "lower": "{:.1%}", "upper": "{:.1%}"}, na_rep="—")

In [ ]:
if len(runs) >= 2:
    baseline_name = "post-trained, unadapted (A)"
    comparison = compare_models(runs, baseline=baseline_name, n_bootstrap=10_000, seed=42)
    display(
        comparison.style.format(
            {
                "accuracy": "{:.1%}",
                "baseline_accuracy": "{:.1%}",
                "difference": "{:+.1%}",
                "lower": "{:+.1%}",
                "upper": "{:+.1%}",
                "probability_positive": "{:.3f}",
            }
        )
    )
else:
    print("At least two evaluated runs are required for a paired comparison.")

In [ ]:
from qwen_text2sql.reporting import figures

if len(runs) >= 2:
    figures.model_comparison(comparison)

### Extending to condition D

Once the base-checkpoint adapter has been trained and evaluated, register its
records in `qwen_text2sql.reporting.context.ARTIFACTS` and add it to `RUNS`
above. The decomposition then reads directly off the comparison table:

- **B − A** — what task adaptation adds to an already post-trained model.
- **B − D** — what general post-training adds, given task adaptation.
- **D − A** — whether task adaptation alone reaches a post-trained model.

## Reading this decomposition

**What it shows.** An attribution of capability across three training stages,
each contrast differing in exactly one respect, with paired intervals.

**What it does not show.**

- **That the stages are additive.** They interact; post-training changes what
  the adapter has to learn, so the differences do not decompose cleanly into
  independent contributions.
- **That Base is worse in general.** It is a statement about this task, this
  prompt and this adapter budget.
- **Anything about a different model family.** One family, one size. Whether the
  pattern holds elsewhere is a separate experiment.

---

### Provenance

Every number above is produced by code in `src/qwen_text2sql/`, which is unit
tested and type checked. This notebook orchestrates and interprets; it does not
define the analysis. If a figure here looks wrong, the fix belongs in `src/` and
comes with a test.